<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/01-deep-learning-foundations-workflow.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Deep Learning Foundations and Workflow** {#deep-learning-foundations-workflow}

Deep learning is often introduced as "machine learning with many neural-network layers." That description identifies a visible architectural property, but it misses the central idea. A deep learning system learns a **composition of representations**: each stage transforms the current description of an example into another description that is more useful for the final objective. Training adjusts these transformations jointly from data.

This chapter develops the vocabulary and workflow used throughout the series. It asks four questions before discussing a particular architecture:

- What exactly is learned by a deep model?
- Why can several nonlinear transformations be more useful than one shallow transformation?
- Which assumptions are built into the architecture, data, objective, and training procedure?
- What evidence is required before a more complex neural model should replace a simpler baseline?

The goal is not to argue that deep learning is always the best tool. It is to understand when learned representations are valuable, what they cost, and how to test that value without confusing training fit with real progress.

### **What Is Deep Learning?** {#what-is-deep-learning}

**Deep learning** is a branch of machine learning that uses multiple parameterized transformations to learn representations and predictions jointly. A feed-forward network can be written as a sequence of hidden states:

$$
\mathbf{h}^{(0)} = \mathbf{x},
$$

$$
\mathbf{z}^{(\ell)}
= W^{(\ell)}\mathbf{h}^{(\ell-1)}+\mathbf{b}^{(\ell)},
\qquad
\mathbf{h}^{(\ell)}
= \phi^{(\ell)}\!\left(\mathbf{z}^{(\ell)}\right),
\quad \ell=1,\ldots,L,
$$

$$
\widehat{\mathbf{y}} = g\!\left(\mathbf{h}^{(L)}\right).
$$

Here, $\mathbf{x}$ is the input; $L$ is the number of learned transformation stages; $W^{(\ell)}$ and $\mathbf{b}^{(\ell)}$ are trainable weights and biases; $\phi^{(\ell)}$ is usually a nonlinear activation or structured operation; $\mathbf{h}^{(\ell)}$ is the representation produced by layer $\ell$; and $g$ converts the final representation into the required output. All trainable quantities are collected into parameters $\theta$.

For supervised learning, training commonly minimizes an empirical objective:

$$
\widehat{\theta}
= \arg\min_{\theta}
\frac{1}{n}\sum_{i=1}^{n}
\mathcal{L}\!\left(f_{\theta}(\mathbf{x}_i),y_i\right)
+ \lambda\,\Omega(\theta).
$$

$n$ is the number of training examples, $f_{\theta}$ is the complete network, $\mathcal{L}$ measures prediction error, $\Omega$ is an optional regularizer, and $\lambda$ controls its strength. The architecture defines which functions can be represented; the objective defines which behavior is preferred; the optimizer defines how parameters are searched; and the data determines which distinctions can be learned. These are different design decisions and should not be collapsed into the word "model."

![A deep neural network maps an input through multiple hidden layers before producing an output.](assets/google-hidden-layers.png){fig-align="center" width="72%" fig-alt="A network diagram with an input layer, two hidden layers, and an output layer."}

*Figure source: Google for Developers, [Machine Learning Glossary: hidden layer](https://developers.google.com/machine-learning/glossary/fundamentals), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*

The word **deep** has no universal layer-count threshold. A network with two hidden layers is deep relative to a linear model, while a modern foundation model may contain hundreds of computational blocks. More importantly, depth should count meaningful transformations, not merely containers in a software printout. A residual block, attention block, recurrent transition, or message-passing step may each contribute depth in a different way.

Deep networks are inspired only loosely by biological nervous systems. Their units, learning rules, data requirements, and connectivity differ substantially from brains. The scientifically useful definition is computational: deep learning learns layered functions and representations by optimizing an objective from data or interaction.

<details>
<summary><strong>NumPy: A minimal two-layer forward pass</strong></summary>

~~~python
import numpy as np

# One example with three input features: shape [input_features].
x = np.array([0.4, -1.2, 0.7])

# The first learned transformation maps 3 inputs to 4 hidden features.
W1 = np.array([
    [0.2, -0.1, 0.4],
    [0.7,  0.3, -0.5],
    [-0.6, 0.2, 0.1],
    [0.1,  0.8, 0.2],
])
b1 = np.zeros(4)

# The output transformation maps 4 hidden features to 2 class logits.
W2 = np.array([
    [0.4, -0.3, 0.2, 0.1],
    [-0.2, 0.5, -0.4, 0.7],
])
b2 = np.zeros(2)

# Forward pass: affine transformation -> nonlinearity -> output scores.
hidden_pre_activation = W1 @ x + b1       # shape [4]
hidden = np.maximum(hidden_pre_activation, 0.0)  # ReLU
logits = W2 @ hidden + b2                 # shape [2]

print("hidden representation:", hidden)
print("class logits:", logits)
~~~

</details>

This code performs inference only. Learning requires a loss, gradients, parameter updates, and held-out evaluation. Those mechanisms are developed in Chapters 4-6. The important point here is that the output is produced through a learned intermediate representation rather than directly from the original features.

**Comparison.** A neural network is an architecture family; deep learning is the broader method of learning layered representations, objectives, and parameters. A large network is not automatically well trained, and a well-trained network is not automatically useful on the intended deployment distribution.

### **From Feature Engineering to Representation Learning** {#feature-engineering-to-representation-learning}

Every learning system needs a representation. An image may begin as pixel intensities, speech as a waveform, a document as token identifiers, and a graph as node and edge attributes. A prediction algorithm cannot operate on the real-world object directly; it operates on a numerical description of that object.

In a classical pipeline, a human designer often specifies a feature map $\psi$ first and trains a comparatively simple predictor afterward:

$$
\mathbf{x}_{\text{raw}}
\xrightarrow{\text{designed }\psi}
\mathbf{r}
\xrightarrow{\text{learned }g_{\omega}}
\widehat{\mathbf{y}}.
$$

Examples include edge histograms for images, MFCC coefficients for speech, TF-IDF vectors for documents, and manually selected ratios for tabular data. These features can be excellent when they encode stable domain knowledge, work with limited data, and make errors easy to inspect.

Representation learning makes the feature map trainable:

$$
\widehat{\mathbf{y}}
= g_{\omega}\!\left(\phi_{\theta}(\mathbf{x})\right).
$$

$\phi_{\theta}$ learns an intermediate representation and $g_{\omega}$ uses it for the task. Because $\theta$ and $\omega$ are optimized together, the representation can preserve distinctions that reduce the final loss. In an image classifier, early layers may respond to local contrast and texture, intermediate layers to motifs or parts, and later layers to task-relevant configurations. In language, representations may move from token identity toward context-dependent syntax, semantics, and discourse information.

<div class="diagram-scroll wide-diagram">

![Feature visualization illustrates a progression from edges and textures to patterns, parts, and object-like features across a trained vision network.](assets/distill-feature-hierarchy.png){fig-align="center" width="100%" fig-alt="Five groups of feature visualizations labelled edges, textures, patterns, parts, and objects."}

</div>

*Figure source: Olah, Mordvintsev, and Schubert, [Feature Visualization](https://distill.pub/2017/feature-visualization/), Distill, 2017, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*

The figure should be interpreted carefully. A feature visualization is an optimized input that strongly activates a unit or channel; it is evidence about model sensitivity, not proof that one neuron literally stores a human concept. Representations are distributed, several features can cooperate, and one feature may respond to multiple patterns. The value of the image is that it makes hierarchical reuse visible: later computations are constructed from earlier learned responses.

Deep learning does not eliminate preprocessing or human judgment. Tokenization, normalization, data augmentation, sampling, label design, and architecture choice still impose structure. "End-to-end" means that a larger portion of the mapping is optimized jointly; it does not mean that the system receives unmediated reality or discovers the correct objective by itself.

| Design | Feature source | Typical strength | Typical limitation |
|---|---|---|---|
| Hand-engineered pipeline | Domain expert and fixed transformation | Data efficiency, transparency, stable behavior | May discard information or require extensive task-specific work |
| Learned representation | Training objective and data | Adapts features to complex high-dimensional signals | Requires evidence, compute, and careful failure analysis |
| Hybrid pipeline | Domain structure plus learned components | Combines constraints with flexible representations | More interfaces and assumptions to validate |

**Example.** A medical-image system may use learned visual features while preserving calibrated acquisition metadata and clinically defined measurements. A fraud system may combine embeddings with explicit transaction rules. Representation learning is therefore a design option, not a demand to remove all engineered knowledge.

**Comparison.** Feature engineering asks, "Which measurements should the predictor receive?" Representation learning asks, "Which intermediate measurements make this objective easier?" Strong systems often answer both questions.

### **Why Depth Can Help** {#why-depth-can-help}

Depth introduces **composition**. A deep model represents a function as

$$
f_{\theta}
= f^{(L)}_{\theta_L}
\circ f^{(L-1)}_{\theta_{L-1}}
\circ \cdots
\circ f^{(1)}_{\theta_1}.
$$

Composition is useful when the target process itself contains reusable stages. A visual decision can combine edges into contours, contours into parts, and parts into configurations. A sentence representation can combine tokens into contextual relations and then into a task-specific decision. The intermediate concepts need not match human labels exactly; the structural advantage is that later computations can reuse earlier ones.

Nonlinearity is essential. If every stage is linear and has no nonlinear activation, then

$$
W^{(3)}W^{(2)}W^{(1)}\mathbf{x}
= W_{\text{equivalent}}\mathbf{x},
$$

so the stack is still one linear transformation. Extra linear layers may change optimization parameterization, but they do not create a nonlinear decision boundary. Activations, attention, gating, normalization interactions, and other nonlinear operations allow the composition to represent richer functions.

<details>
<summary><strong>NumPy: Several linear layers collapse into one</strong></summary>

~~~python
import numpy as np

rng = np.random.default_rng(4)
x = rng.normal(size=3)
W1 = rng.normal(size=(5, 3))
W2 = rng.normal(size=(4, 5))
W3 = rng.normal(size=(2, 4))

layered_output = W3 @ (W2 @ (W1 @ x))
equivalent_weight = W3 @ W2 @ W1
single_layer_output = equivalent_weight @ x

assert np.allclose(layered_output, single_layer_output)
print(layered_output)
~~~

</details>

A shallow network with enough width can approximate a broad class of functions. This universal approximation result does **not** imply that depth is irrelevant. The required shallow network may be impractically wide, may fail to expose reusable structure, or may be harder to learn from finite data. Depth-separation results construct functions that compact deep networks can represent but substantially shallower networks require exponentially more units to approximate. [Telgarsky's depth result](https://proceedings.mlr.press/v49/telgarsky16.html) is one formal example.

That result is an existence statement, not a guarantee that adding layers improves every dataset. Greater depth can also make optimization unstable, increase latency, amplify data shortcuts, and create unnecessary capacity. Residual connections, normalization, initialization, and modern optimizers help make deep networks trainable, but they do not replace task evidence.

| Question | Shallow model | Deeper model |
|---|---|---|
| Can it represent nonlinear functions? | Yes, with nonlinear units and enough width | Yes |
| Can it reuse intermediate computations? | Limited hierarchy | Natural compositional hierarchy |
| Is optimization automatically easier? | Often simpler | Can be harder without architectural support |
| Is it automatically more accurate? | No | No |
| When is depth most plausible? | Simple or low-data relationships | Structured, high-dimensional, compositional signals |

**Comparison.** Width supplies parallel features; depth supplies sequential reuse and composition. Real architectures balance both, and validation evidence determines whether the added structure is useful.

### **Inductive Bias and Compositional Structure** {#inductive-bias-compositional-structure}

An **inductive bias** is an assumption that favors some solutions over others before all possible input-output cases have been observed. Generalization is impossible without such preferences: many functions can fit a finite training set perfectly while disagreeing everywhere else.

Deep learning is sometimes described as assumption-free because features are learned. In reality, every architecture contains strong assumptions:

| Architecture | Important structural bias | Suitable structure | Possible mismatch |
|---|---|---|---|
| Multilayer perceptron | Flexible global mixing after vectorization | Fixed-size feature vectors | Ignores spatial, temporal, or graph structure unless encoded manually |
| Convolutional network | Local connectivity and translation-related weight sharing | Images, grids, local signals | Global interactions require depth or additional mechanisms |
| Recurrent network | Shared state transition across time | Ordered sequences and streaming data | Sequential computation limits parallelism and long-range memory |
| Transformer | Content-dependent pairwise interaction with shared blocks | Contextual sequences and multimodal tokens | Standard attention can be expensive for long contexts |
| Graph neural network | Permutation-aware neighborhood aggregation | Relational and geometric data | Similar local neighborhoods can become indistinguishable |
| State-space model | Learned state dynamics with efficient sequence scanning | Long sequences and streaming | Compression into state may lose interactions needed by the task |

Bias also enters through data augmentation, loss functions, optimizers, regularization, and sampling. Horizontal image flips assert that the label is usually unchanged under reflection. Causal masking asserts that a prediction cannot inspect future tokens. Contrastive learning asserts which transformed examples should share a representation. A large model trained end-to-end still inherits every one of these decisions.

The practical question is therefore not whether a model is biased, but whether its biases match the problem. Convolution is useful when local patterns recur across positions. It is less appropriate when absolute position carries the meaning and translation should change the label. A graph model is useful when edges express real relations; adding arbitrary edges can inject the wrong invariance.

**Application.** For satellite land-cover classification, local spatial patterns and approximate translation invariance make a CNN plausible. For transaction fraud, time order, account relations, and changing behavior may require a hybrid of sequence, graph, and explicit rule-based components. Choosing "a deep model" without identifying the structure is not yet model design.

**Comparison.** Capacity describes how many behaviors a model can express. Inductive bias describes which behaviors are easier to express or learn. More capacity cannot compensate reliably for a badly matched bias.

### **The Data-Model-Compute Relationship** {#data-model-compute-relationship}

Modern deep learning is shaped by an interaction among **data**, **model capacity**, and **compute**. Improving only one axis eventually exposes a bottleneck in another.

**Data** supplies the distinctions the model can observe. Relevant dimensions include quantity, diversity, label quality, provenance, recency, duplication, and coverage of important subgroups. A billion repeated or systematically biased examples do not equal a billion independent observations. Data transformations and sampling determine which evidence is emphasized during training.

**Model capacity** determines which mappings and representations are available. A model that is too restricted underfits; a highly flexible model can memorize noise, exploit shortcuts, or exceed deployment constraints. Parameter count is only one capacity indicator. Architecture, context length, sparsity, routing, precision, and optimization all affect usable capacity.

**Compute** determines which experiments and training trajectories are feasible. It includes accelerator operations, memory, communication, data loading, energy, engineering time, and wall-clock time. More compute may permit a larger model, more data, a longer search, or more robust evaluation. Spending it on one choice means not spending it on another.

An empirical scaling law summarizes observed behavior over a specified model family, dataset, objective, and compute range. It is not a universal physical law. The Chinchilla study showed that, under its language-model setting and fixed training budget, balancing model size with substantially more training data outperformed simply making the model larger. The durable lesson is resource balance, not a timeless constant that applies unchanged to every modality or deployment. See [Hoffmann et al., Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556).

| Bottleneck | Observable symptom | Better first response |
|---|---|---|
| Insufficient or mismatched data | Large slice errors, unstable validation, shortcut learning | Improve collection, labels, splits, and coverage |
| Insufficient model capacity | Training and validation performance both plateau poorly | Add appropriate capacity or a better architecture |
| Excessive capacity for evidence | Training improves while validation degrades or varies strongly | Regularize, simplify, augment, or obtain more evidence |
| Insufficient compute | Training stops early, experiments are underpowered | Reduce model, improve efficiency, or narrow the question |
| Poor objective | Metric improves without improving the real decision | Redesign labels, loss, constraints, or evaluation |

The final row matters because data, model, and compute optimize the objective they are given. They do not determine whether the objective represents user value, scientific truth, fairness, or safety. Problem formulation remains a human and organizational responsibility.

**Comparison.** Scaling can improve a well-posed learning process, but it also scales data defects, objective mismatch, and operational cost. Better results come from balancing evidence, capacity, computation, and the actual decision being supported.

### **Problem Formulations and Application Domains** {#problem-formulations-application-domains}

Deep learning is not one task. A task formulation specifies the input unit, output structure, supervision, objective, evaluation protocol, and deployment decision. The same architecture can support several formulations, and the same formulation can be solved by several architectures.

| Formulation | Input -> output | Example | Typical output layer or objective |
|---|---|---|---|
| Classification | object -> class distribution | image diagnosis category | logits with cross-entropy |
| Regression | object -> continuous value | remaining useful life | scalar/vector with regression loss |
| Dense prediction | spatial input -> spatial labels | semantic segmentation | per-location logits |
| Sequence labeling | sequence -> label per position | speech frame or token annotation | per-step logits, sometimes structured decoding |
| Retrieval/metric learning | query and candidates -> similarity/ranking | image-text search | embedding similarity or contrastive loss |
| Autoregressive generation | prefix -> next-element distribution | text, audio, or image-token generation | conditional likelihood |
| Reconstruction/self-supervision | transformed input -> missing/original information | masked image modeling | reconstruction or representation objective |
| Sequential control | observations -> actions | robotics or game playing | value, policy, or actor-critic objectives |

Application domains include computer vision, speech, language, recommendation, scientific modeling, healthcare, finance, robotics, cybersecurity, and multimodal interaction. The domain name does not determine the formulation. A medical image can be classified, segmented, retrieved, generated, or combined with text. Each version has different labels, risks, and evidence requirements.

Architecture and task should also remain separate. A Transformer is not synonymous with language modeling; it can encode images, forecast time series, process biological sequences, or fuse modalities. A CNN is not synonymous with classification; it can produce dense maps or intermediate features. Clear separation prevents a common mistake: choosing a fashionable architecture before specifying what output the system must produce.

Before training, write a compact task contract:

1. What constitutes one example at prediction time?
2. Which information is legally and operationally available then?
3. What output is required, and how will it change a decision?
4. How are labels or learning signals produced?
5. Which mistakes have the highest cost?
6. Which distribution shift should the evaluation simulate?
7. What latency, memory, privacy, and safety constraints apply?

**Comparison.** A model maps numerical inputs to outputs. A deep learning system includes the data process, learning objective, evaluation, inference path, and decision that give that mapping meaning.

### **When Deep Learning Is Not the Right Tool** {#when-deep-learning-not-right-tool}

Deep learning is most compelling when inputs are high-dimensional, useful features are difficult to specify, reusable representations can be learned from substantial data, and nonlinear structure justifies the additional complexity. Outside those conditions, a simpler method may be better.

| Situation | Why deep learning may be a poor first choice | Candidate alternative |
|---|---|---|
| Small structured tabular dataset | Neural estimates may be unstable and offer little representational advantage | Linear model, tree ensemble, Bayesian model |
| Exact rules define correctness | Approximation adds unnecessary error | Deterministic program, parser, constraint solver |
| Very limited labels and no suitable pretraining | Capacity exceeds available supervision | Domain features, probabilistic model, active learning |
| Strict interpretability or audit requirements | Internal representations may be difficult to justify | Sparse model, scoring rule, monotonic model |
| Hard latency, memory, or energy budget | Model may not fit the serving environment | Heuristic, compact classical model, lookup or cache |
| Rapidly changing target with delayed labels | Expensive retraining may lag behind the process | Rules plus monitoring, online or adaptive methods |
| Causal intervention question | Predictive fit alone does not identify intervention effects | Experimental or causal inference design |

A simple model is also a diagnostic instrument. If a linear classifier already meets the operational target, a larger model must justify its maintenance, inference, monitoring, and failure costs. If a heuristic outperforms the neural model, the problem may lie in data or objective design rather than insufficient architecture.

Pretrained models change the calculation because they allow representation learning to be amortized over a much larger source dataset. Even then, transfer can fail through domain shift, inherited bias, licensing constraints, privacy concerns, or inference cost. "Use a pretrained model" is a hypothesis to evaluate, not an exemption from evaluation.

**Application.** For a few thousand clean rows of customer attributes, gradient-boosted trees are a strong baseline. For millions of product images with complex visual variation, learned visual representations are more plausible. A production system may use both: a neural encoder supplies embeddings while a simpler calibrated model makes the final constrained decision.

**Comparison.** The correct question is not "Can a neural network fit this dataset?" It usually can. The question is whether it creates enough reliable value to justify its evidence and operational burden.

### **A Reproducible Deep Learning Workflow** {#reproducible-deep-learning-workflow}

A reliable workflow separates problem definition, model selection, and final evaluation. One practical sequence is:

$$
\text{decision contract}
\rightarrow \text{data audit and split}
\rightarrow \text{baseline}
\rightarrow \text{representation and model}
\rightarrow \text{training}
\rightarrow \text{validation and diagnosis}
\rightarrow \text{locked test evaluation}
\rightarrow \text{deployment and monitoring}.
$$

**Decision contract.** Define the prediction unit, available information, desired action, error costs, and operational constraints. This prevents optimizing an easy proxy that does not improve the intended outcome.

**Data audit and split.** Inspect provenance, labels, duplicates, missingness, subgroup coverage, and temporal structure. Split before fitting normalization or feature transformations. Group or chronological splits are often more realistic than random rows.

**Baseline.** Begin with a majority rule, heuristic, linear model, or small established architecture. Google's [Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml) emphasizes that a simple first model provides baseline behavior and exposes infrastructure problems before architectural complexity is added.

**Training and validation.** Optimize parameters only with the training split. Use validation evidence for architecture, hyperparameters, stopping, and threshold choices. Keep the test set isolated until those choices are fixed.

**Reproducibility record.** Preserve code revision, environment, data version, split identifiers, random seeds, configuration, logs, and the selected checkpoint. PyTorch notes that exact equality is not guaranteed across releases, platforms, or CPU/GPU execution even with identical seeds; reproducibility claims must therefore state the environment and determinism settings. See the official [PyTorch reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness).

The following case study compares a linear baseline with a two-hidden-layer MLP on a nonlinear two-moons problem. It deliberately keeps the dataset and network small so every stage remains visible.

<details>
<summary><strong>PyTorch: Data construction, splitting, and model definitions</strong></summary>

~~~python
from copy import deepcopy
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

SEED = 17


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_two_moons(
    n_samples: int = 1_500,
    noise: float = 0.18,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    """Create a nonlinear two-class dataset without external packages."""
    rng = np.random.default_rng(seed)
    n_outer = n_samples // 2
    n_inner = n_samples - n_outer

    outer_angle = rng.uniform(0.0, np.pi, n_outer)
    inner_angle = rng.uniform(0.0, np.pi, n_inner)
    outer = np.column_stack((np.cos(outer_angle), np.sin(outer_angle)))
    inner = np.column_stack((1.0 - np.cos(inner_angle), 0.5 - np.sin(inner_angle)))

    features = np.vstack((outer, inner))
    labels = np.concatenate((
        np.zeros(n_outer, dtype=np.int64),
        np.ones(n_inner, dtype=np.int64),
    ))
    features += rng.normal(0.0, noise, features.shape)

    order = rng.permutation(n_samples)
    return features[order].astype(np.float32), labels[order]


def split_and_standardize(
    features: np.ndarray,
    labels: np.ndarray,
) -> dict[str, tuple[torch.Tensor, torch.Tensor]]:
    """Split first; fit preprocessing statistics on training data only."""
    train_end = int(0.60 * len(features))
    validation_end = int(0.80 * len(features))
    raw_splits = {
        "train": (features[:train_end], labels[:train_end]),
        "validation": (features[train_end:validation_end], labels[train_end:validation_end]),
        "test": (features[validation_end:], labels[validation_end:]),
    }

    train_features = raw_splits["train"][0]
    mean = train_features.mean(axis=0, keepdims=True)
    std = train_features.std(axis=0, keepdims=True).clip(min=1e-6)

    return {
        name: (
            torch.tensor((x_values - mean) / std, dtype=torch.float32),
            torch.tensor(y_values, dtype=torch.long),
        )
        for name, (x_values, y_values) in raw_splits.items()
    }


class LinearBaseline(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.classifier = nn.Linear(2, 2)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(inputs)


class SmallMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.network(inputs)


seed_everything(SEED)
features, labels = make_two_moons()
splits = split_and_standardize(features, labels)
~~~

</details>

<details>
<summary><strong>PyTorch: Training, checkpoint selection, and evaluation</strong></summary>

~~~python
@torch.inference_mode()
def accuracy(model, split) -> float:
    features, labels = split
    predictions = model(features).argmax(dim=1)
    return (predictions == labels).float().mean().item()


def train_model(
    model,
    splits,
    learning_rate: float = 0.03,
    maximum_epochs: int = 800,
    patience: int = 80,
):
    """Train on train data and select the checkpoint by validation loss."""
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    train_features, train_labels = splits["train"]
    validation_features, validation_labels = splits["validation"]
    history = {"train_loss": [], "validation_loss": []}
    best_state = deepcopy(model.state_dict())
    best_validation_loss = float("inf")
    epochs_without_improvement = 0

    for _ in range(maximum_epochs):
        # 1. Update parameters using training examples only.
        model.train()
        optimizer.zero_grad()
        train_logits = model(train_features)
        train_loss = loss_function(train_logits, train_labels)
        train_loss.backward()
        optimizer.step()

        # 2. Measure validation loss without updating parameters.
        model.eval()
        with torch.inference_mode():
            validation_logits = model(validation_features)
            validation_loss = loss_function(validation_logits, validation_labels)

        history["train_loss"].append(train_loss.item())
        history["validation_loss"].append(validation_loss.item())

        # 3. Preserve the checkpoint that generalizes best to validation data.
        if validation_loss.item() < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss.item()
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, history


models = {}
histories = {}
for name, model in {
    "Linear baseline": LinearBaseline(),
    "Two-hidden-layer MLP": SmallMLP(),
}.items():
    seed_everything(SEED)
    trained_model, history = train_model(model, splits)
    models[name] = trained_model
    histories[name] = history

for name, model in models.items():
    print(
        name,
        "validation=", round(accuracy(model, splits["validation"]), 4),
        "test=", round(accuracy(model, splits["test"]), 4),
    )
~~~

</details>

The run used to generate this chapter produced:

| Model | Selected epoch | Validation accuracy | Test accuracy |
|---|---:|---:|---:|
| Linear baseline | 98 | 0.873 | 0.887 |
| Two-hidden-layer MLP | 54 | 0.977 | 0.963 |

The exact selected epoch can vary when software or hardware changes, which is why the workflow preserves the checkpoint and environment rather than treating one number as timeless. The performance difference is structural: a linear classifier can draw only one straight boundary, while the nonlinear MLP can bend its boundary around the two moons.

![The linear baseline uses a straight decision boundary, while the MLP learns a nonlinear boundary that follows the two moons. Yellow rings mark test errors.](assets/dl01-linear-vs-mlp.png){fig-align="center" width="100%" fig-alt="Side-by-side decision boundaries for a linear classifier and a two-hidden-layer MLP on the two-moons dataset."}

*Figure generated locally by the reproducible PyTorch experiment in this chapter.*

This example does not prove that the MLP is universally superior. It demonstrates a narrower claim: when the decision structure is nonlinear and there is enough evidence, a model with nonlinear intermediate representations can remove a limitation visible in the baseline.

**Comparison.** A training script is not yet a reproducible experiment. Reproducibility requires a fixed question, immutable splits, recorded configuration, validation-based decisions, a locked test evaluation, and artifacts that allow the run to be reconstructed.

### **Baselines, Error Analysis, and Iteration** {#baselines-error-analysis-iteration}

A **baseline** establishes the minimum behavior a new method must improve. Useful baselines include:

- a random or majority-class predictor that checks metric interpretation;
- a domain heuristic that represents existing knowledge;
- a linear or tree-based model that tests whether complex representation learning is needed;
- a small neural model that isolates gains from scale;
- the current production system, including its latency and failure behavior;
- an ablation that removes the component claimed to create the improvement.

Beating a weak baseline is weak evidence. If a proposed architecture is compared only with random prediction, the experiment does not show that its complexity is necessary. Comparisons should use the same data split, preprocessing information, evaluation protocol, and tuning budget whenever possible.

Aggregate metrics are the beginning of error analysis, not the end. Errors should be sliced by conditions that represent model hypotheses or deployment risks: class, subgroup, source, time, input length, noise, confidence, missing features, or proximity to a decision boundary.

For the two-moons experiment, define an overlap band around standardized feature 2:

~~~python
@torch.inference_mode()
def slice_report(model, split):
    features, labels = split
    predictions = model(features).argmax(dim=1)

    # This band contains points near the region where a straight boundary fails.
    overlap_mask = features[:, 1].abs() < 0.45

    return {
        "overall": (predictions == labels).float().mean().item(),
        "overlap_band": (
            (predictions[overlap_mask] == labels[overlap_mask]).float().mean().item()
        ),
        "outside_band": (
            (predictions[~overlap_mask] == labels[~overlap_mask]).float().mean().item()
        ),
    }


for name, model in models.items():
    print(name, slice_report(model, splits["test"]))
~~~

| Model | Overall | Overlap band | Outside band |
|---|---:|---:|---:|
| Linear baseline | 0.887 | 0.705 | 0.951 |
| Two-hidden-layer MLP | 0.963 | 1.000 | 0.951 |

The slice reveals more than the aggregate improvement. Both models perform similarly outside the selected band, while the MLP's gain is concentrated where the linear assumption is wrong. That result supports the proposed mechanism. If the gain had appeared only in an unrelated easy region, the explanation would need revision.

A practical error taxonomy is:

| Error source | Diagnostic evidence | Typical next action |
|---|---|---|
| Data or label error | Contradictory duplicates, uncertain annotation | Repair data or preserve uncertainty |
| Coverage error | Poor performance on a source or subgroup | Collect representative examples or constrain use |
| Representation/architecture error | Consistent failure on a structural pattern | Change inductive bias or representation |
| Optimization error | High training loss, unstable gradients | Inspect scale, initialization, optimizer, and numerics |
| Generalization error | Low training loss but weak validation | Regularize, simplify, augment, or add evidence |
| Objective error | Metric rises while useful behavior worsens | Redesign loss, labels, constraints, or decision rule |
| Evaluation error | Leakage, repeated test tuning, wrong slices | Rebuild the protocol before changing the model |
| Deployment error | Offline/online mismatch, latency or drift | Validate the full serving path and monitor it |

Iteration should follow evidence:

$$
\text{observe failure}
\rightarrow \text{form a mechanism-level hypothesis}
\rightarrow \text{change one relevant factor}
\rightarrow \text{run a controlled comparison}
\rightarrow \text{update the error map}.
$$

Changing architecture, augmentation, optimizer, data, and metric simultaneously may improve a score, but it prevents learning why. Controlled experiments turn model development into cumulative knowledge rather than an expensive sequence of guesses.

**Comparison.** Optimization asks how to reduce the training objective. Error analysis asks whether the remaining failures come from data, representation, optimization, objective, evaluation, or deployment. The second question determines what should change next.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Deep learning is best understood as a method for learning composed representations under explicit architectural, data, objective, and computational assumptions. Its advantage is not that humans stop designing the system. Its advantage is that many useful intermediate features can be adapted jointly to evidence and reused across complex transformations.

| Dimension | Classical feature-based approach | Deep representation-learning approach |
|---|---|---|
| Intermediate features | Primarily specified before fitting | Primarily learned with the objective |
| Typical model | Linear, kernel, tree, probabilistic model | Layered differentiable architecture |
| Data requirement | Often effective with smaller structured datasets | Often benefits from large or pretrained datasets |
| Compute and engineering | Usually lower | Potentially much higher |
| Structural assumptions | Feature design and model family | Architecture, objective, data, and training process |
| Debugging | Features and coefficients may be easier to inspect | Requires representation, slice, and failure analysis |
| Best use | Clear features, limited data, strict constraints | High-dimensional signals and reusable complex structure |

The main conclusions are:

1. A deep network is a composition of learned transformations, not merely a large collection of parameters.
2. Representation learning shifts part of feature design into the optimization process but does not remove preprocessing, task definition, or human assumptions.
3. Depth can represent compositional functions efficiently, but extra layers are not automatically useful and linear layers without nonlinear operations still collapse to one linear map.
4. Architecture is an inductive bias. CNNs, recurrent models, Transformers, graph networks, and state-space models favor different structures.
5. Data, model capacity, compute, and objective must be balanced. Scaling one does not repair defects in the others.
6. Deep learning should compete against credible simple baselines and must justify its operational cost.
7. A trustworthy workflow separates training, validation, and test decisions, records reproducibility information, and iterates through error hypotheses rather than architecture fashion.

The next chapter develops the tensor operations, computation graphs, and PyTorch abstractions used to express these layered transformations precisely.
